# Theme-9: Logarithmic Regret Bounding via UCB in Collaborative Filtering
## Analysis Notebook — MovieLens 100K

This notebook walks through the full pipeline interactively:
1. Data loading and EDA
2. UCB-OCO model training
3. SGD-MF baseline training
4. Metric evaluation
5. Regret analysis and convergence visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.load_data import get_train_test_split, build_interaction_matrix
from src.mf import train_ucb_oco, predict_all
from src.baseline import SGDMatrixFactorization
from src.evaluate import rmse, mae, precision_at_k, recall_at_k, ndcg_at_k, convergence_rate
from src.plots import *

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Data Loading & EDA

In [ ]:
train_df, test_df, n_users, n_items = get_train_test_split(test_size=0.2, random_state=42)
print(f'Users: {n_users}, Items: {n_items}')
print(f'Train: {len(train_df)}, Test: {len(test_df)}')
train_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution (Train)')
axes[0].set_xlabel('Rating')

ratings_per_user = train_df.groupby('user_id').size()
axes[1].hist(ratings_per_user, bins=40, color='tomato', edgecolor='white')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Number of Ratings')
plt.tight_layout()

## 2. UCB-OCO Training

In [ ]:
ucb_model, ucb_losses = train_ucb_oco(
    train_df, n_users, n_items,
    n_factors=20, eta0=0.05, lambda_reg=0.01, c_ucb=1.5
)

## 3. SGD-MF Baseline Training

In [ ]:
sgd_model = SGDMatrixFactorization(
    n_users=n_users, n_items=n_items,
    n_factors=20, lr=0.005, lambda_reg=0.02, n_epochs=20
).fit(train_df)

## 4. Evaluation

In [ ]:
K = 10
ucb_preds, actuals = predict_all(ucb_model, test_df)
sgd_preds = np.array([sgd_model.predict(r.user_id, r.item_id) for r in test_df.itertuples(index=False)])

results = {
    'RMSE':         [rmse(ucb_preds, actuals), rmse(sgd_preds, actuals)],
    'MAE':          [mae(ucb_preds, actuals),  mae(sgd_preds, actuals)],
    f'Precision@{K}': [precision_at_k(ucb_model, test_df, train_df, K),
                       precision_at_k(sgd_model, test_df, train_df, K)],
    f'Recall@{K}':    [recall_at_k(ucb_model, test_df, train_df, K),
                       recall_at_k(sgd_model, test_df, train_df, K)],
    f'NDCG@{K}':      [ndcg_at_k(ucb_model, test_df, train_df, K),
                       ndcg_at_k(sgd_model, test_df, train_df, K)],
}

pd.DataFrame(results, index=['UCB-OCO', 'SGD-MF']).round(4)

## 5. Convergence & Regret Analysis

In [ ]:
# Convergence rate: should be ~0.5 for O(1/sqrt(t))
alpha = convergence_rate(ucb_losses)
print(f'Empirical convergence rate alpha = {alpha:.4f}  (theory: 0.5 for O(1/sqrt(t)))')

In [ ]:
# Cumulative regret on log-log scale
T = len(ucb_losses)
t = np.arange(1, T + 1)
cum_regret = np.cumsum(ucb_losses)
ref = cum_regret[0] * np.sqrt(t * np.log(t + 1)) / (np.sqrt(1 * np.log(2)))

plt.figure(figsize=(9, 4))
plt.loglog(t, cum_regret, color='steelblue', lw=1.5, label='UCB-OCO Cumulative Loss')
plt.loglog(t, ref, color='gray', lw=1, ls='--', label=r'$O(\sqrt{T \ln T})$ reference')
plt.xlabel('Round t (log scale)')
plt.ylabel('Cumulative Loss (log scale)')
plt.title('Logarithmic Regret Bound Verification')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# UCB decomposition: exploitation vs exploration
top_n = 30
items = np.arange(top_n)
mu = ucb_model.mu_hat[:top_n]
var = ucb_model.M2[:top_n] / ucb_model.N[:top_n]
exploration = ucb_model.c_ucb * np.sqrt(var / ucb_model.N[:top_n] + np.log(ucb_model.t + 1) / ucb_model.N[:top_n])

plt.figure(figsize=(11, 4))
plt.bar(items, mu, label='Exploitation (mean reward)', color='steelblue', alpha=0.85)
plt.bar(items, exploration, bottom=mu, label='Exploration bonus', color='orange', alpha=0.85)
plt.xlabel('Item ID')
plt.ylabel('UCB Score')
plt.title('UCB Score Decomposition: Exploitation vs Exploration')
plt.legend()
plt.tight_layout()
plt.show()